# Data Cleaning

Step-by-step data cleaning for the Phishing Detection dataset.

In [7]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv(r'G:/My Drive/URL-Phish_Dataset.csv')
print(f"Original shape: {df.shape}")
df.head()

Original shape: (116600, 26)


,url,url_len,dom,dom_len,is_ip,tld,tld_len,subdom_cnt,letter_cnt,digit_cnt,...,under_cnt,letter_ratio,digit_ratio,spec_ratio,is_https,slash_cnt,entropy,path_len,query_len,label
0,https://www.rmit.edu.au/,24,rmit.edu.au,11,0,edu.au,6,1,17,0,...,0,0.708333,0.0,0.291667,1,3,3.709148,1,0,0
1,http://www.latrobe.edu.au/,26,latrobe.edu.au,14,0,edu.au,6,1,19,0,...,0,0.730769,0.0,0.269231,0,3,3.738149,1,0,0
2,https://www.cqu.edu.au/,23,cqu.edu.au,10,0,edu.au,6,1,16,0,...,0,0.695652,0.0,0.304348,1,3,3.609668,1,0,0
3,http://bond.edu.au/,19,bond.edu.au,11,0,edu.au,6,0,13,0,...,0,0.684211,0.0,0.315789,0,3,3.576618,1,0,0
4,http://www.csu.edu.au/,22,csu.edu.au,10,0,edu.au,6,1,15,0,...,0,0.681818,0.0,0.318182,0,3,3.503998,1,0,0


## Drop Text Columns

`url`, `dom`, `tld` are raw text — not usable as numerical features directly.

In [8]:
df = df.drop(columns=['url', 'dom', 'tld'])
print(f"After dropping text columns: {df.shape}")
print(f"Columns: {list(df.columns)}")

After dropping text columns: (116600, 23)
Columns: ['url_len', 'dom_len', 'is_ip', 'tld_len', 'subdom_cnt', 'letter_cnt', 'digit_cnt', 'special_cnt', 'eq_cnt', 'qm_cnt', 'amp_cnt', 'dot_cnt', 'dash_cnt', 'under_cnt', 'letter_ratio', 'digit_ratio', 'spec_ratio', 'is_https', 'slash_cnt', 'entropy', 'path_len', 'query_len', 'label']


## Remove Rows with Missing Values

Only 14 missing values across 116K+ rows — safe to drop those rows entirely.

In [9]:
# Show which column has missing values
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nTotal rows before: {len(df):,}")

df = df.dropna().reset_index(drop=True)

print(f"Total rows after:  {len(df):,}")
print(f"Remaining nulls:   {df.isnull().sum().sum()}")

Missing values per column:
Series([], dtype: int64)

Total rows before: 116,600
Total rows after:  116,600
Remaining nulls:   0


## Drop Highly Correlated Features

Removing features with a correlation > 0.7 to reduce multicollinearity. For each pair of highly correlated features, we keep the one that has a lower average correlation with all other features.

In [10]:
# Compute correlation matrix (excluding label)
feature_cols = [c for c in df.columns if c != 'label']
corr_matrix = df[feature_cols].corr().abs()

# Find pairs with correlation > 0.7
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

cols_to_drop = set()
threshold = 0.7

for col in upper.columns:
    correlated = upper.index[upper[col] > threshold].tolist()
    for corr_col in correlated:
        mean_corr_col = corr_matrix[col].drop(col).mean()
        mean_corr_corr = corr_matrix[corr_col].drop(corr_col).mean()
        if mean_corr_col > mean_corr_corr:
            cols_to_drop.add(col)
        else:
            cols_to_drop.add(corr_col)

cols_to_drop = sorted(cols_to_drop)
print(f"Dropping {len(cols_to_drop)} highly correlated features:")
for c in cols_to_drop:
    print(f"  - {c}")

df = df.drop(columns=cols_to_drop)
print(f"\nDataset shape after dropping: {df.shape}")

Dropping 5 highly correlated features:
  - entropy
  - eq_cnt
  - letter_cnt
  - special_cnt
  - url_len

Dataset shape after dropping: (116600, 18)


## Stratified Train-Test Split

Splitting the dataset into 80% training and 20% testing sets, maintaining the 85/15 ratio of Legitimate to Phishing URLs.

In [11]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['label'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train set: {X_train.shape[0]:,} samples")
print(f"Test set:  {X_test.shape[0]:,} samples")

# Verify stratification
print("\nTrain label distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTest label distribution:")
print(y_test.value_counts(normalize=True).round(4))

Train set: 93,280 samples
Test set:  23,320 samples

Train label distribution:
label
0    0.8576
1    0.1424
Name: proportion, dtype: float64

Test label distribution:
label
0    0.8576
1    0.1424
Name: proportion, dtype: float64


## Save Cleaned Data

Saving the final cleaned sets into a `data` folder.

In [12]:
import os

os.makedirs('data', exist_ok=True)

train_df = pd.concat([X_train, y_train], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_csv('data/train.csv', index=False)
test_df.to_csv('data/test.csv', index=False)
df.to_csv('data/cleaned_full.csv', index=False)

print("✅ Saved files:")
print(f"  - data/train.csv         ({train_df.shape})")
print(f"  - data/test.csv          ({test_df.shape})")
print(f"  - data/cleaned_full.csv  ({df.shape})")

✅ Saved files:
  - data/train.csv         ((93280, 18))
  - data/test.csv          ((23320, 18))
  - data/cleaned_full.csv  ((116600, 18))
